# 네이버 영화리뷰 감성 분석

### 01. 데이터 로드

**✅ 실습 과제**
- [네이버 영화 리뷰 데이터셋](https://raw.githubusercontent.com/e9t/nsmc/master/ratings.txt)을 불러온다.
- 데이터의 컬럼 구성과 샘플을 확인한다.

**🔍 확인 질문**
1. 리뷰 텍스트와 정답 라벨은 각각 어떤 컬럼에 저장되어 있는가?
2. 긍정 / 부정 라벨은 어떤 값으로 표현되어 있는가?

In [ ]:
# !pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.5/223.5 MB 20.9 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 15.9 MB/s  0:00:00
  Attempting uninstall: h5py
    Found existing installation: h5py 3.15.1
    Uninstalling h5py-3.15.1:
      Successfully uninstalled h5py-3.15.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [tensorflow]2 [tensorflow]


In [3]:
# 데이터 다운로드
from tensorflow.keras.utils import get_file

ratings_train_path = get_file("ratings_train.txt", "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt")
ratings_test_path = get_file("ratings_test.txt", "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt")

ratings_train_path, ratings_test_path

14628807/14628807 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
4893335/4893335 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


('/Users/areum/.keras/datasets/ratings_train.txt',
 '/Users/areum/.keras/datasets/ratings_test.txt')

In [ ]:
# 데이터프레임 생성
import pandas as pd

train_df = pd.read_table(ratings_train_path)
test_df = pd.read_table(ratings_test_path)

train_df.head(20)

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1
5,5403919,막 걸음마 뗀 3세부터 초등학교 1학년생인 8살용영화.ㅋㅋㅋ...별반개도 아까움.,0
6,7797314,원작의 긴장감을 제대로 살려내지못했다.,0
7,9443947,별 반개도 아깝다 욕나온다 이응경 길용우 연기생활이몇년인지..정말 발로해도 그것보단...,0
8,7156791,액션이 없는데도 재미 있는 몇안되는 영화,1
9,5912145,왜케 평점이 낮은건데? 꽤 볼만한데.. 헐리우드식 화려함에만 너무 길들여져 있나?,1


label
- 0 = 부정
- 1 = 긍정

In [ ]:
train_df['label'].value_counts()

label
0    75173
1    74827
Name: count, dtype: int64

In [30]:
# 데이터 샘플링
df = train_df.sample(40000, random_state=42)
df['label'].value_counts()

label
0    20240
1    19760
Name: count, dtype: int64

In [31]:
sample_df = test_df.sample(20000, random_state=42)
sample_df['label'].value_counts()

label
1    10080
0     9920
Name: count, dtype: int64

### 02. 데이터 전처리

##### 02-01. 한글 전처리

**✅ 실습 과제**
- 특수문자, 숫자, 불필요한 기호를 제거한다.
- 정규표현식을 사용하여 한글만 남긴다.

**🔍 확인 질문**
1. 한글 전처리를 하지 않으면 어떤 문제가 발생할 수 있는가?
2. 감성 분석에서 이모지나 느낌표는 제거하는 것이 항상 옳은가?




In [32]:
# 결측치 제거
display(df.isnull().sum())

df = df.dropna(how='any')

id          0
document    2
label       0
dtype: int64

In [33]:
df.isnull().sum()

id          0
document    0
label       0
dtype: int64

In [34]:
# 한글 토큰화 전처리 (특수문자 처리, 어간 추출, 불용어 처리) -> 함수
from tqdm import tqdm
from konlpy.tag import Okt

stop_words = ['의','가','이','은','들','는','좀','잘','걍','과','도','를','으로','자','에','와','한','하다']
sentences = []

okt = Okt()

df['document'] = df['document'].replace(r'[^0-9가-힣ㄱ-ㅎㅏ-ㅣ\s]','', regex=True)

for sentence in tqdm(df['document']):
    tokens = okt.morphs(sentence, stem=True)                           
    tokens = [token for token in tokens if token not in stop_words]   
    sentences.append(tokens)

100%|██████████| 39998/39998 [00:59<00:00, 676.78it/s]


##### 02-02. Tokenizing & Sequencing

**✅ 실습 과제**
- Tokenizer를 사용해 단어 사전을 생성한다.
- 문장을 정수 시퀀스로 변환한다.
- padding을 적용하여 시퀀스 길이를 맞춘다.

**🔍 확인 질문**
1. `num_words` 파라미터는 어떤 역할을 하는가?
2. padding을 하지 않으면 배치 학습에서 어떤 문제가 발생하는가?



In [ ]:
# sequence 작업 (단어사전 생성, 텍스트-수열 변환)
from torchtext.vocab import build_vocab_from_iterator

# 값을 하나씩 반환하는 함수
def yield_tokens(data):
    for tokens in data:
        yield tokens

In [ ]:
# padding 작업

##### 02-03. Sequence Decoding

**✅ 실습 과제**
- 정수 시퀀스를 다시 텍스트로 복원해본다.
- 토큰 인덱스와 단어의 매핑 관계를 확인한다.

**🔍 확인 질문**
1. `<OOV>` 토큰은 언제 사용되는가?
2. 디코딩 결과가 원문과 완전히 동일하지 않은 이유는 무엇인가?


### 03. 모델 생성 및 학습

**✅ 실습 과제**
- Embedding Layer를 포함한 감성분석 모델을 정의한다.
- 손실 함수와 옵티마이저를 설정한다.
- 학습 과정을 통해 loss 변화를 확인한다.

**🔍 확인 질문**
1. Embedding Layer는 어떤 역할을 하는가?
2. One-hot 인코딩 대신 Embedding을 사용하는 이유는 무엇인가?

In [ ]:
# 모델 정의

In [ ]:
# 모델 인스턴스 생성

In [ ]:
# 학습

### 04. 모델 평가

**✅ 실습 과제**
- 검증 데이터로 모델 성능을 평가한다.
- 정확도(acc)와 손실(loss)을 확인한다.

**🔍 확인 질문**
1. 학습 데이터 성능과 평가 데이터 성능 차이가 의미하는 것은 무엇인가?
2. 과적합 여부는 어떻게 판단할 수 있는가?

### 05. 모델 추론

**✅ 실습 과제**
- 임의의 문장을 입력하여 감성을 예측한다.
- 출력 확률을 기반으로 긍/부정을 해석한다.

**🔍 확인 질문**
1. 모델 출력값은 확률인가 점수인가?
2. 기준값(threshold)은 왜 필요한가?